# Phase 12 — Standalone: Cell 29b (Positions-Scan)

Kontrolle zu Cell 29: derselbe Blindheits-Eingriff (Query sieht nur sich selbst,
nur in den 10 Voll-Attention-Layern) an der Ziel-Position 43, am Köder-Attribut 42
und an sechs gleichmäßig verteilten Kontroll-Positionen. Selbstversorgend —
**frische Runtime**, dann nur diese Zelle ausführen. Laufzeit ~25–35 min (eager).

In [ ]:
# === Cell 29b — POSITIONS-SCAN der totalen Blindheit ========================
# Das "Ohr" ist zweistufig: Transport der ' local'-Information nach Pos 43
# (Attention/Delta-Pfad, nie untersucht) -> Detektor (Router-Cluster+Experten,
# lokalisiert). Dieser Test schneidet TRANSPORT-KANTEN in den VOLL-Attention-
# Layern der Hybrid-Architektur (Gated-DeltaNet-Layer ignorieren Attention-
# Masken - was dort laeuft, koennen wir nicht kantenscharf schneiden; genau
# deshalb ist der "alles"-Arm der Interpretations-Anker).
# Methode: eigener 4D-Mask-Pfad im Prefill (Query q darf Key k nicht sehen),
# dann manuelles Sampling aus dem "tauben" KV-Cache. Arme (N=32):
#   none        pure Kausal-4D (Baseline + Kalibrier-Gate: muss der normalen
#               Vorhersage entsprechen, sonst Abbruch)
#   43<-42      DIE Kante (Vervollstaendigung liest das Attribut)
#   43<-41      Kontroll-Key  ("'s")   - Key-Spezifitaet
#   44<-42      Kontroll-Query (",")   - Query-Spezifitaet
#   43<-alles   Query 43 sieht keine Vergangenheit - Anker: toetet nicht mal
#               das, laeuft die Komposition im Delta-Pfad, Einzelkanten moot.
# Braucht das INSTRUCT-Modell (Cell 3) + PROMPTS.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, numpy as np
# ---------------- Selbstversorgung: Modell + Prompts sicherstellen ----------
import glob, json, gc
for _n in ("model_b","tok_b"):                  # Base-Reste aus Cell 28 raus
    if _n in globals():
        try: del globals()[_n]
        except Exception: pass
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError(("GPU nicht leer genug (%.1f GB frei, ~45 noetig): vermutlich "
        "belegt noch ein frueheres Modell den Speicher. Loesung: Laufzeit -> "
        "Sitzung neu starten, dann NUR diese Zelle ausfuehren.")%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
N_ANS=32; MAX_NEW=24; CHUNK=8
SCAFF="<|im_start|>user\n"
def think_prefix(u,th=""):
    return SCAFF+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n"+th+"\n</think>\n\n"
# ---------------- pure Logik (testbar) --------------------------------------
def build_mask4d(L,edges,blockall_q=None):
    """(1,1,L,L)-Additivmaske: kausal 0/-inf; edges=[(q,k),...] zusaetzlich
       -inf; blockall_q: diese Query sieht nur sich selbst. numpy-float32."""
    M=np.zeros((L,L),dtype=np.float32)
    M[np.triu_indices(L,1)]=-np.inf
    for q,k in edges: M[q,k]=-np.inf
    if blockall_q is not None:
        M[blockall_q,:blockall_q]=-np.inf
    return M[None,None]
def twoprop(k1,n1,k2,n2):
    p=(k1+k2)/(n1+n2); se=math.sqrt(p*(1-p)*(1/n1+1/n2)) if 0<p<1 else 0.0
    if se==0: return 1.0
    z=abs(k1/n1-k2/n2)/se
    return 2*(1-0.5*(1+math.erf(z/math.sqrt(2))))
def verdict_ear(kn,kall,ke,kc1,kc2,N):
    kill=lambda k:(k<kn and twoprop(k,N,kn,N)<0.05)
    if not kill(kall): return "DELTA-PFAD"
    if kill(ke) and not (kill(kc1) or kill(kc2)): return "OHRKANAL"
    if kill(ke): return "UNSPEZIFISCH"
    return "VERTEILT"
def tok_span(offs,c0,c1):
    return [i for i,(s,e) in enumerate(offs) if e>c0 and s<c1 and e>s]
# ---------------- Klassifikator ---------------------------------------------
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),(0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will would can it on as at be by".split())
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[ch for ch in t if ch.isalpha()]
    fo=[ch for ch in al if ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW)]
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def wilson(k,n,z=1.96):
    if n==0: return (0.0,0.0,0.0)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n)
    h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
# ---------------- Architektur-Recon -----------------------------------------
full_attn=[]; delta=[]
for name,mod in model.named_modules():
    m=re.fullmatch(r"model\.layers\.(\d+)\.self_attn",name)
    if m: full_attn.append(int(m.group(1)))
    m=re.fullmatch(r"model\.layers\.(\d+)\.linear_attn",name)
    if m: delta.append(int(m.group(1)))
print("Hybrid-Architektur: %d Voll-Attention-Layer %s | %d Delta-Layer"
      %(len(full_attn),full_attn[:8],len(delta)))
assert full_attn, "keine Voll-Attention-Layer gefunden - Kanten-Test nicht moeglich"
try:
    model.set_attn_implementation("eager")
except Exception:
    try: model.config._attn_implementation="eager"
    except Exception: pass
print("Attention-Implementierung:",getattr(model.config,"_attn_implementation","?"),
      "(eager erzwungen: alle Arme laufen im selben Kernel-Universum)")
# ---------------- Prompt + Spannen ------------------------------------------
TAB=PROMPTS[[p for p in PROMPT_IDS if p.startswith("643fdf5d")][0]]
prefix=think_prefix(TAB,"")
enc=tokenizer(prefix,return_offsets_mapping=True)
IDS=enc["input_ids"]; L=len(IDS)
c0=len(SCAFF)+TAB.index("local name"); c1=c0+len("local name")
DEC=tok_span(enc["offset_mapping"],c0,c1)
Q,K=DEC[-1],DEC[0]                                     # 43 <- 42
print("Koeder-Span %s | Kante: Query %d (%r) <- Key %d (%r)"
      %(DEC,Q,tokenizer.decode([IDS[Q]]),K,tokenizer.decode([IDS[K]])))
dev=model.device; DT=next(model.parameters()).dtype
ids_t=torch.tensor([IDS],device=dev)
# ---- v5: modell-eigene Maske ABFANGEN und nur die Kante zusetzen -----------
# (v3/v4 scheiterten identisch: eine handgebaute "pure Kausal"-Maske ersetzte
#  die modell-eigene, die offenbar NICHT pur kausal ist. v5 modifiziert statt
#  zu ersetzen - der none-Arm ist damit per Konstruktion der Normalfall.)
ATTN=[mod for name,mod in model.named_modules()
      if re.fullmatch(r"model\.layers\.\d+\.self_attn",name)]
HK={"edges":None,"blockq":None,"cap":{}}
def apply_edges(M,edges,blockq):
    M=M.clone()
    if M.dtype==torch.bool:
        for q,k in (edges or []): M[...,q,k]=False
        if blockq is not None: M[...,blockq,:blockq]=False
    else:
        v=torch.finfo(M.dtype).min/2
        for q,k in (edges or []): M[...,q,k]=v
        if blockq is not None: M[...,blockq,:blockq]=v
    return M
def _fallback_mask(qlen,b):
    M=torch.from_numpy(build_mask4d(qlen,[])).to(device=dev,dtype=torch.float32)
    return torch.nan_to_num(M,neginf=torch.finfo(DT).min/2).to(DT).expand(b,-1,-1,-1)
def _pre(mod,args,kwargs):
    hs=kwargs.get("hidden_states", args[0] if (args and torch.is_tensor(args[0])) else None)
    if hs is None or hs.shape[1]<=1: return None
    am=kwargs.get("attention_mask",None)
    if "info" not in HK["cap"]:
        HK["cap"]["info"]=(type(am).__name__,
                           tuple(am.shape) if torch.is_tensor(am) else None,
                           str(getattr(am,"dtype",None)))
        if torch.is_tensor(am) and am.dim()==4 and am.shape[-2]==hs.shape[1]:
            if am.dtype.is_floating_point:
                thr=torch.finfo(am.dtype).min/4
                row=lambda q:int((am[0,0,q]>thr).sum())
            else:
                row=lambda q:int(am[0,0,q].sum())
            HK["cap"]["open"]={int(q):row(q) for q in (5,Q,hs.shape[1]-1) if q<hs.shape[1]}
    if HK["edges"] is None and HK["blockq"] is None: return None
    qlen=hs.shape[1]
    if not (torch.is_tensor(am) and am.dim()==4 and am.shape[-2]==qlen):
        if torch.is_tensor(am) or am is None:
            am=_fallback_mask(qlen,hs.shape[0])
            HK["cap"]["fallback"]=True
        else:
            return None
    kwargs["attention_mask"]=apply_edges(am,HK["edges"],HK["blockq"])
    return (args,kwargs)
hooks=[a.register_forward_pre_hook(_pre,with_kwargs=True) for a in ATTN]
@torch.no_grad()
def gen_masked(edges,blockq,n,max_new):
    outs=[]
    HK["edges"],HK["blockq"]=edges,blockq
    for s in range(0,n,CHUNK):
        b=min(CHUNK,n-s)
        inp=ids_t.repeat(b,1)
        out=model(input_ids=inp,use_cache=True)
        past=out.past_key_values
        tok=torch.multinomial(torch.softmax(out.logits[:,-1].float(),-1),1)
        seq=[tok]
        for _ in range(max_new-1):
            out=model(input_ids=tok,past_key_values=past,use_cache=True)
            past=out.past_key_values
            tok=torch.multinomial(torch.softmax(out.logits[:,-1].float(),-1),1)
            seq.append(tok)
        S=torch.cat(seq,1)
        outs+=[tokenizer.decode(r,skip_special_tokens=True) for r in S]
    HK["edges"],HK["blockq"]=None,None
    return outs
# ---------------- Kalibrier-Gate --------------------------------------------
@torch.no_grad()
def _logits(edges,blockq):
    HK["edges"],HK["blockq"]=edges,blockq
    l=model(input_ids=ids_t).logits[0,-1].float()
    HK["edges"],HK["blockq"]=None,None
    return l
lp=_logits(None,None); lc=_logits([],None)
d_causal=float((lp-lc).abs().max())
lh=_logits([],L-1)
d_heavy=float((lp-lh).abs().max())
print("NATUERLICHE MASKE in self_attn:",HK["cap"].get("info"),
      "| offene Keys je Query:",HK["cap"].get("open"),
      "| Fallback benutzt:",HK["cap"].get("fallback",False))
d_block=float((lh-lc).abs().max())
print("KALIBRIERUNG: Kernel-Offset (maskiert vs. maskenlos) = %.4f (informativ)"%d_causal)
print("              Blockade vs. gleicher Pfad = %.2f (muss gross)"%d_block)
if d_block<=1.0:
    for h in hooks: h.remove()
assert d_block>1.0, "Blockade wirkungslos im selben Kernel-Pfad - Maske erreicht self_attn nicht, Abbruch"
# ---------------- Arme ------------------------------------------------------
# ---------------- Positions-Scan: ist die Blindheit bei 43 spezifisch? ------
# Cell 29 zeigte: Query 43 komplett blind (nur sich selbst sehend) toetet den
# Kipp. Offen blieb, ob das POSITIONSSPEZIFISCH ist oder ob Blindheit an JEDER
# inhaltstragenden Stelle den Kipp killt (generische Stoerung). Dieser Scan
# wiederholt exakt denselben Eingriff an Kontroll-Positionen im User-Text.
def pick_controls(lo,hi,avoid,n=6):
    """n gleichmaessig verteilte Positionen in [lo,hi], ohne avoid"""
    cand=[p for p in range(lo,hi+1) if p not in avoid]
    if len(cand)<=n: return cand
    idx=np.linspace(0,len(cand)-1,n).round().astype(int)
    return sorted({cand[int(i)] for i in idx})
def verdict_scan(kn,kq,kctrl,N):
    kill=lambda k:(k<kn and twoprop(k,N,kn,N)<0.05)
    nk=sum(1 for k in kctrl if kill(k))
    if not kill(kq): return "KEIN-EFFEKT"
    if nk==0: return "POSITIONSSPEZIFISCH"
    if nk>=len(kctrl)/2.0: return "GENERISCH"
    return "ANGEREICHERT"
UT=tok_span(enc["offset_mapping"],len(SCAFF),len(SCAFF)+len(TAB))
AVOID={K-1,K,Q,Q+1}
CPOS=pick_controls(UT[0],UT[-1],AVOID,6)
SCAN=[("none",None)]+[("blind@%d"%p,p) for p in sorted(set([Q,K]+CPOS))]
SW=("takeover","gloss","latin-switch(fr)")
print("\nPOSITIONS-SCAN (totale Blindheit je Query, N=%d):"%N_ANS)
print("  Ziel Q=%d %r | Koeder-Attribut K=%d %r | Kontrollen %s"
      %(Q,tokenizer.decode([IDS[Q]]),K,tokenizer.decode([IDS[K]]),CPOS))
R29B={}
for name,pos in SCAN:
    cls=[classify_answer(x) for x in gen_masked([],pos,N_ANS,MAX_NEW)]
    k=sum(1 for c in cls if c in SW)
    R29B[name]=(k,N_ANS,dict(collections.Counter(cls)))
    p,lo,hi=wilson(k,N_ANS)
    tag=""
    if pos==Q: tag="  <- ZIEL"
    elif pos==K: tag="  <- Koeder-Attribut"
    tok=repr(tokenizer.decode([IDS[pos]])) if pos is not None else "(Baseline)"
    print("  %-11s %-14s rate=%5.1f%% [%4.1f,%4.1f]  %s%s"
          %(name,tok,100*p,100*lo,100*hi,R29B[name][2],tag))
    if name=="none" and k<2:
        for h in hooks: h.remove()
        raise RuntimeError(("UNIVERSUM VERZERRT: Baseline kippt nicht mehr (%d/32) - "
            "Vergleiche waeren bedeutungslos. Abbruch.")%k)
for _h in hooks: _h.remove()
kn=R29B["none"][0]; kq=R29B["blind@%d"%Q][0]; kk=R29B["blind@%d"%K][0]
kctrl=[R29B["blind@%d"%p][0] for p in CPOS]
rate=lambda k:k/N_ANS
order=sorted([(R29B["blind@%d"%p][0],p) for p in sorted(set([Q,K]+CPOS))])
rank=[p for _,p in order].index(Q)+1
print("\n  Rangliste (niedrigste Kipp-Rate zuerst): %s"
      %", ".join("%d:%d/32"%(p,k) for k,p in order))
print("  Ziel-Position %d belegt Rang %d von %d | p(Ziel vs. Baseline)=%.4f"
      %(Q,rank,len(order),twoprop(kq,N_ANS,kn,N_ANS)))
print("  Koeder-Attribut %d: %d/32 (p=%.3f) | Kontrollen median %d/32"
      %(K,kk,twoprop(kk,N_ANS,kn,N_ANS),int(np.median(kctrl))))
code=verdict_scan(kn,kq,kctrl,N_ANS)
print("\nVERDIKT:",end=" ")
if code=="POSITIONSSPEZIFISCH":
    print("POSITIONSSPEZIFISCH: nur die Blindheit an der Vervollstaendigungs-")
    print("  Position %d toetet (%d/32 von %d/32); keine Kontroll-Position tut es"%(Q,kq,kn))
    print("  (median %d/32). Der Transport ist damit auch auf der Attention-Stufe"%int(np.median(kctrl)))
    print("  lokalisiert: Position 43 muss ihre Vergangenheit lesen koennen,")
    print("  andere Positionen nicht. Kette Phrase -> Attention -> Router -> Kipp.")
elif code=="ANGEREICHERT":
    nk=sum(1 for k in kctrl if (k<kn and twoprop(k,N_ANS,kn,N_ANS)<0.05))
    print("ANGEREICHERT: Ziel toetet (%d/32), aber auch %d von %d Kontrollen -"%(kq,nk,len(kctrl)))
    print("  die Blindheit wirkt teils generisch; die Ziel-Position ist nur")
    print("  angereichert, nicht exklusiv. Rangliste oben ansehen.")
elif code=="GENERISCH":
    print("GENERISCH: Blindheit toetet an den meisten Positionen (Kontrollen median")
    print("  %d/32 bei Baseline %d/32) - der Eingriff stoert die Prompt-Verarbeitung"%(int(np.median(kctrl)),kn))
    print("  breit. Aus Cell 29 folgt dann NUR 'Voll-Attention ist noetig', nichts")
    print("  ueber die Koeder-Position. Ehrliche Abschwaechung des Vorbefunds.")
else:
    print("KEIN-EFFEKT: die Ziel-Blindheit repliziert nicht (%d/32 bei Baseline %d/32)"%(kq,kn))
    print("  - Cell 29 war ein Zufallstreffer oder die Session unterscheidet sich;")
    print("  Zahlen oben ansehen, keine Aussage.")
SCAN_RESULTS=dict(arms={k:v[:2] for k,v in R29B.items()},verdict=code,
                  target=Q,attr=K,controls=CPOS,rank=rank)
